# Day 09. Exercise 00
# Regularization

## 0. Imports

In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [85]:
df = pd.read_csv('../../datasets/dayofweek.csv')
df.head()


,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,-0.788667,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,-0.756764,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,-0.724861,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,-0.692958,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,4,-0.661055,-2.562352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [86]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [87]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=21, fit_intercept=False)


In [88]:
%%time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
val_scores = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model.fit(X_fold_train, y_fold_train)
    
    train_pred = model.predict(X_fold_train)
    val_pred = model.predict(X_fold_val)
    
    train_acc = accuracy_score(y_fold_train, train_pred)
    val_acc = accuracy_score(y_fold_val, val_pred)
    
    train_scores.append(train_acc)
    val_scores.append(val_acc)
    
    print(f"train -  {train_acc:.5f}   |   valid -  {val_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(val_scores):.5f}")
print(f"Std is {np.std(val_scores):.5f}")

train -  0.64056   |   valid -  0.65926
train -  0.63561   |   valid -  0.62222
train -  0.64468   |   valid -  0.60000
train -  0.64056   |   valid -  0.64444
train -  0.65375   |   valid -  0.60741
train -  0.62902   |   valid -  0.60000
train -  0.66117   |   valid -  0.60000
train -  0.63726   |   valid -  0.54074
train -  0.63756   |   valid -  0.66418
train -  0.64745   |   valid -  0.61194
Average accuracy on crossval is 0.61502
Std is 0.03399
CPU times: user 3.14 s, sys: 15.6 ms, total: 3.16 s
Wall time: 355 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [89]:
model = LogisticRegression(penalty=None, random_state=21, fit_intercept=False, max_iter=1000)
model.fit(X_train, y_train)
print("None:", model.score(X_test, y_test))

None: 0.650887573964497


In [90]:
model = LogisticRegression(penalty='l1', solver='liblinear', random_state=21, fit_intercept=False)
model.fit(X_train, y_train)
print("L1:", model.score(X_test, y_test))

L1: 0.6183431952662722


In [91]:
model = LogisticRegression(penalty='l2', random_state=21, fit_intercept=False)
model.fit(X_train, y_train)
print("L2:", model.score(X_test, y_test))

L2: 0.6331360946745562


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [92]:
from sklearn.svm import SVC

model = SVC(probability=True, kernel='linear', random_state=21)

In [93]:
skf = StratifiedKFold(n_splits=10, shuffle=False, random_state=None)

train_scores = []
val_scores = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    val_pred = model.predict(X_fold_val)

    train_acc = accuracy_score(y_fold_train, train_pred)
    val_acc = accuracy_score(y_fold_val, val_pred)
    
    train_scores.append(train_acc)
    val_scores.append(val_acc)
    
    print(f"train -  {train_acc:.5f}   |   valid -  {val_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(val_scores):.5f}")
print(f"Std is {np.std(val_scores):.5f}")

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222


train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [94]:
model = SVC(kernel='linear', C=0.1, random_state=21)
model.fit(X_train, y_train)
print(f"C=0.1 | Test accuracy: {model.score(X_test, y_test):.5f}")

C=0.1 | Test accuracy: 0.59763


In [95]:
model = SVC(kernel='linear', C=1, random_state=21)
model.fit(X_train, y_train)
print(f"C=1   | Test accuracy: {model.score(X_test, y_test):.5f}")

C=1   | Test accuracy: 0.71598


In [96]:
model = SVC(kernel='linear', C=10, random_state=21)
model.fit(X_train, y_train)
print(f"C=10  | Test accuracy: {model.score(X_test, y_test):.5f}")

C=10  | Test accuracy: 0.74556


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [97]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=10, random_state=21)

In [98]:
skf = StratifiedKFold(n_splits=10, shuffle=False, random_state=None)

train_scores = []
val_scores = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    val_pred = model.predict(X_fold_val)

    train_acc = accuracy_score(y_fold_train, train_pred)
    val_acc = accuracy_score(y_fold_val, val_pred)
    
    train_scores.append(train_acc)
    val_scores.append(val_acc)
    
    print(f"train -  {train_acc:.5f}   |   valid -  {val_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(val_scores):.5f}")
print(f"Std is {np.std(val_scores):.5f}")

train -  0.81039   |   valid -  0.74074
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.76296
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.75373
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [99]:
from sklearn.model_selection import cross_val_score

param_combinations = [
    {'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 4},
    {'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2},
    {'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
]

print("\nTesting parameter combinations:")
for params in param_combinations:
    model = DecisionTreeClassifier(random_state=21, **params)
    scores = cross_val_score(model, X_train, y_train, cv=10, scoring='accuracy')
    print(f"{params} | Accuracy: {np.mean(scores):.5f} ± {np.std(scores):.5f}")


Testing parameter combinations:
{'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 4} | Accuracy: 0.54153 ± 0.02660
{'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2} | Accuracy: 0.71067 ± 0.03255
{'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1} | Accuracy: 0.85459 ± 0.02682


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [100]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)

skf = StratifiedKFold(n_splits=10, shuffle=False, random_state=None)

train_scores = []
val_scores = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    model.fit(X_fold_train, y_fold_train)

    train_pred = model.predict(X_fold_train)
    val_pred = model.predict(X_fold_val)

    train_acc = accuracy_score(y_fold_train, train_pred)
    val_acc = accuracy_score(y_fold_val, val_pred)
    
    train_scores.append(train_acc)
    val_scores.append(val_acc)
    
    print(f"train -  {train_acc:.5f}   |   valid -  {val_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(val_scores):.5f}")
print(f"Std is {np.std(val_scores):.5f}")

train -  0.96455   |   valid -  0.88148
train -  0.96208   |   valid -  0.91852
train -  0.96785   |   valid -  0.86667
train -  0.96455   |   valid -  0.89630
train -  0.96538   |   valid -  0.91111
train -  0.96538   |   valid -  0.88148
train -  0.97115   |   valid -  0.91852
train -  0.96867   |   valid -  0.85185
train -  0.97364   |   valid -  0.88060
train -  0.97941   |   valid -  0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [101]:
max_depths = [5, 20, None]
n_estimators_list = [100, 200]

print("Testing Random Forest parameters:")
for depth in max_depths:
    for n_est in n_estimators_list:
        model = RandomForestClassifier(
            max_depth=depth,
            n_estimators=n_est,
            random_state=21,
        )
        scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        print(f"max_depth={str(depth).ljust(4)} | n_estimators={n_est:3} | "
              f"Accuracy: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

Testing Random Forest parameters:
max_depth=5    | n_estimators=100 | Accuracy: 0.58015 ± 0.02453
max_depth=5    | n_estimators=200 | Accuracy: 0.56901 ± 0.01237
max_depth=20   | n_estimators=100 | Accuracy: 0.89539 ± 0.01403
max_depth=20   | n_estimators=200 | Accuracy: 0.89391 ± 0.01425
max_depth=None | n_estimators=100 | Accuracy: 0.90058 ± 0.01195
max_depth=None | n_estimators=200 | Accuracy: 0.90281 ± 0.01146


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [102]:
best_model = RandomForestClassifier(max_depth=None, n_estimators=200)
best_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200)

In [103]:
y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)
print(f"Final test accuracy: {final_accuracy:.5f}")

Final test accuracy: 0.93195


In [104]:
error_rates = {}
max_error = 0
day_max_error = 0

for day in best_model.classes_:
    day_mask = (y_test == day)
    total_samples = sum(day_mask)
    if total_samples > 0:
        wrong_predictions = sum(y_pred[day_mask] != day)
        error_rate = wrong_predictions / total_samples * 100
        error_rates[day] = error_rate
        if error_rate > max_error:
            max_error = error_rate
            day_max_error = day

print(f"day when model makes the most errors is {day_max_error} - {max_error:.1f} %")

day when model makes the most errors is 0 - 25.9 %


In [105]:
import joblib

joblib.dump(best_model, 'best_model.joblib')
print("Модель сохранена как 'best_model.joblib'")

Модель сохранена как 'best_model.joblib'
